<a href="https://colab.research.google.com/github/qnra/MEDICA/blob/main/L7_Galkowski_35790x2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
imie = "Konrad"
nazwisko = "Galkowski"
id_20 = 1+35790%30
print("Imie: ", imie)
print("Nazwisko: ", nazwisko)
print("Id_20: ", id_20)

Imie:  Konrad
Nazwisko:  Galkowski
Id_20:  1


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, stddev, count

In [ ]:
spark = SparkSession.builder.appName("Galkowski").getOrCreate()

In [ ]:
df_wykonanie = spark.read.csv("execution.csv", header=True, inferSchema=True, sep=';')
df_osoba = spark.read.csv("person.csv", header=True, inferSchema=True, sep=';')
df_dzial = spark.read.csv("dept.csv", header=True, inferSchema=True, sep=';')
df_stanowisko = spark.read.csv("position.csv", header=True, inferSchema=True, sep=';')
df_czynnosc = spark.read.csv("action.csv", header=True, inferSchema=True, sep=';')

In [ ]:
df_wykonanie.show(5)

df_osoba.show(5)

df_dzial.show(5)

df_stanowisko.show(5)

df_czynnosc.show(5)

+---+-----+---------+-------+-----------+------------+----+----+-------+
| id|id_20|id_person|id_dept|id_position|id_execution|time|cost|payment|
+---+-----+---------+-------+-----------+------------+----+----+-------+
|  1|    0|        5|      4|          1|           3| 590| 487|   1384|
|  2|    1|        6|      2|          3|           1| 563| 873|    455|
|  3|    2|        1|      1|          3|           3| 160| 659|   1009|
|  4|    3|        4|      6|          1|           2| 101| 848|   1850|
|  5|    4|        1|      2|          1|           4| 476| 750|   1401|
+---+-----+---------+-------+-----------+------------+----+----+-------+
only showing top 5 rows
+---+-----+
| id| name|
+---+-----+
|  1|  Ela|
|  2|  Jan|
|  3|  Ola|
|  4|Marek|
|  5|  Ula|
+---+-----+
only showing top 5 rows
+---+---------+
| id|     dept|
+---+---------+
|  1|       IT|
|  2|  Service|
|  3|  Testing|
|  4|Technical|
|  5|   Coding|
+---+---------+
only showing top 5 rows
+---+-----------+
|

In [ ]:
df_joined = df_wykonanie.join(df_osoba, df_wykonanie.id_person == df_osoba.id) \
    .join(df_dzial, df_wykonanie.id_dept == df_dzial.id) \
    .join(df_stanowisko, df_wykonanie.id_position == df_stanowisko.id) \
    .join(df_czynnosc, df_wykonanie.id_execution == df_czynnosc.id) \
    .select(
        df_wykonanie.id_20,
        df_wykonanie.time.alias("czas"),
        df_wykonanie.cost.alias("koszt"),
        df_wykonanie.payment.alias("zaplata"),
        df_osoba.name.alias("imie"),
        df_dzial.dept.alias("dzial"),
        df_stanowisko.position.alias("stanowisko"),
        df_czynnosc.action.alias("czynnosc")
    )

In [ ]:
df_filtered = df_joined.filter(col("id_20") == 1)
df_filtered.show(5)

+-----+----+-----+-------+-----+---------+-----------+------------+
|id_20|czas|koszt|zaplata| imie|    dzial| stanowisko|    czynnosc|
+-----+----+-----+-------+-----+---------+-----------+------------+
|    1| 563|  873|    455| Olek|  Service|coordinator|organization|
|    1| 344|  674|   1338|  Ula|Technical|   employee|organization|
|    1| 259|  260|    404|  Ola|Technical|   employee|     testing|
|    1| 296|  701|    625|Marek|  Graphic|   employee|    planning|
|    1| 595|  458|   1619|  Ola|Technical|coordinator|     testing|
+-----+----+-----+-------+-----+---------+-----------+------------+
only showing top 5 rows


In [ ]:
stats = df_filtered.select(
    count("*").alias("ilosc_obserwacji"),
    mean("czas").alias("srednia_czas"),
    stddev("czas").alias("odchylenie_czas"),
    mean("koszt").alias("srednia_koszt"),
    stddev("koszt").alias("odchylenie_koszt"),
    mean("zaplata").alias("srednia_zaplata"),
    stddev("zaplata").alias("odchylenie_zaplata")
)
stats.show()

+----------------+------------------+-----------------+------------------+------------------+---------------+------------------+
|ilosc_obserwacji|      srednia_czas|  odchylenie_czas|     srednia_koszt|  odchylenie_koszt|srednia_zaplata|odchylenie_zaplata|
+----------------+------------------+-----------------+------------------+------------------+---------------+------------------+
|              30|341.73333333333335|185.0310194559808|458.73333333333335|234.73211926809668|         1065.4|505.13293572941825|
+----------------+------------------+-----------------+------------------+------------------+---------------+------------------+



ETAP 3 ------------------

zapis polaczonego widoku do csv

In [ ]:
output_path = "execution_full_v2"

df_joined.coalesce(1).write \
    .option("header", True) \
    .option("sep", ";") \
    .mode("overwrite") \
    .csv(output_path)


ponowne wczytanie danych

In [ ]:
df_v2 = spark.read.csv(
    output_path,
    header=True,
    inferSchema=True,
    sep=';'
)

df_v2.show(5)


+-----+----+-----+-------+-----+---------+-----------+------------+
|id_20|czas|koszt|zaplata| imie|    dzial| stanowisko|    czynnosc|
+-----+----+-----+-------+-----+---------+-----------+------------+
|    0| 590|  487|   1384|  Ula|Technical|    manager|      coding|
|    1| 563|  873|    455| Olek|  Service|coordinator|organization|
|    2| 160|  659|   1009|  Ela|       IT|coordinator|      coding|
|    3| 101|  848|   1850|Marek|  Graphic|    manager|    planning|
|    4| 476|  750|   1401|  Ela|  Service|    manager|     testing|
+-----+----+-----+-------+-----+---------+-----------+------------+
only showing top 5 rows


ANALIZA TYLKO NA DFv2


In [ ]:
df_v2_filtered = df_v2.filter(col("id_20") == id_20)

df_v2_filtered.show(5)


+-----+----+-----+-------+-----+---------+-----------+------------+
|id_20|czas|koszt|zaplata| imie|    dzial| stanowisko|    czynnosc|
+-----+----+-----+-------+-----+---------+-----------+------------+
|    1| 563|  873|    455| Olek|  Service|coordinator|organization|
|    1| 344|  674|   1338|  Ula|Technical|   employee|organization|
|    1| 259|  260|    404|  Ola|Technical|   employee|     testing|
|    1| 296|  701|    625|Marek|  Graphic|   employee|    planning|
|    1| 595|  458|   1619|  Ola|Technical|coordinator|     testing|
+-----+----+-----+-------+-----+---------+-----------+------------+
only showing top 5 rows


statystyki koncowe

In [ ]:
final_stats = df_v2_filtered.select(
    count("*").alias("ilosc_obserwacji"),
    mean("czas").alias("srednia_czas"),
    stddev("czas").alias("odchylenie_czas"),
    mean("koszt").alias("srednia_koszt"),
    stddev("koszt").alias("odchylenie_koszt"),
    mean("zaplata").alias("srednia_zaplata"),
    stddev("zaplata").alias("odchylenie_zaplata")
)

final_stats.show()


+----------------+------------------+-----------------+------------------+------------------+---------------+------------------+
|ilosc_obserwacji|      srednia_czas|  odchylenie_czas|     srednia_koszt|  odchylenie_koszt|srednia_zaplata|odchylenie_zaplata|
+----------------+------------------+-----------------+------------------+------------------+---------------+------------------+
|              30|341.73333333333335|185.0310194559808|458.73333333333335|234.73211926809668|         1065.4|505.13293572941825|
+----------------+------------------+-----------------+------------------+------------------+---------------+------------------+



zapis wyniku

In [ ]:
df_v2_filtered.coalesce(1).write \
    .option("header", True) \
    .option("sep", ";") \
    .mode("overwrite") \
    .csv("wyniki_etap3")
